# Notebook 4: Motif Utility and Robustness

This notebook tests whether clean motif flows are useful for improving predictions on hard examples and corrupted inputs.

It compares five models:
- backbone baseline
- backbone + frozen full-motif hybrid
- backbone + frozen top-motif hybrid
- backbone + joint full-motif hybrid
- backbone + joint top-motif hybrid

The notebook is meant to answer a practical question: after discovering motif families in `z`, do those motif activations actually help the model make better decisions, especially on hard examples and corrupted inputs?


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")
IN_COLAB = "google.colab" in sys.modules
LOCAL_REPO_DIR = Path.cwd().resolve()
if not (LOCAL_REPO_DIR / "configs" / "flow").exists() and (LOCAL_REPO_DIR.parent / "configs" / "flow").exists():
    LOCAL_REPO_DIR = LOCAL_REPO_DIR.parent

if REPO_DIR.parent.exists() and IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = REPO_DIR if REPO_DIR.exists() else LOCAL_REPO_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch

try:
    from flow_circuits.data import build_cifar10_splits
    from flow_circuits.evaluation import (
        run_motif_clean_utility_experiment,
        run_motif_corruption_utility_experiment,
    )
    from flow_circuits.training import load_components_from_checkpoint, load_yaml_config
except ModuleNotFoundError:
    REPO_DIR = Path("/content/model_interpretability")
    if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(Path.cwd())], check=True)
    from flow_circuits.data import build_cifar10_splits
    from flow_circuits.evaluation import (
        run_motif_clean_utility_experiment,
        run_motif_corruption_utility_experiment,
    )
    from flow_circuits.training import load_components_from_checkpoint, load_yaml_config


## Parameters


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
HARD_PAIRS_TOP_K = 3
TRIGGER_MODE = "hard_pair_top2_and_low_margin"
MARGIN_QUANTILE = 0.25
TOP_MOTIF_FRACTION = 0.25
MIN_TOP_MOTIFS = 5
MAX_TOP_MOTIFS = 20
FIT_MAX_IMAGES = 4000
VAL_MAX_IMAGES = 1000
TEST_MAX_IMAGES = 1000
CORRUPTION_NAMES = ["gaussian_noise", "gaussian_blur", "contrast"]
CORRUPTION_SEVERITIES = [1, 3, 5]
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
NB03_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_z_motif_discovery_and_analysis" / CONFIG_NAME
FROZEN_MOTIF_ARTIFACT = NB03_OUTPUT_DIR / "frozen" / "frozen_clean_motifs.json"
JOINT_MOTIF_ARTIFACT = NB03_OUTPUT_DIR / "joint" / "joint_clean_motifs.json"
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb04_motif_utility_and_robustness" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
motif_artifacts = {
    "frozen": json.loads(FROZEN_MOTIF_ARTIFACT.read_text(encoding="utf-8")),
    "joint": json.loads(JOINT_MOTIF_ARTIFACT.read_text(encoding="utf-8")),
}

print("=" * 72)
print("Notebook 4 Workflow")
print("=" * 72)
print("What this notebook is doing:")
print("  1. Build motif activation features from the clean motifs found in nb03.")
print("  2. Learn pair-specific motif readouts on fit/val data.")
print("  3. Test whether motif-based hybrids beat the backbone on hard examples.")
print("  4. Reuse the same clean motifs on corrupted inputs to test robustness.")
print()
print("How to read the main metrics:")
print("  backbone_overall_accuracy: plain classifier accuracy with no motif correction.")
print("  full_motif_overall_accuracy: accuracy when all motif activations can correct hard cases.")
print("  top_motif_overall_accuracy: same idea, but using only the most useful motif subset.")
print("  trigger_accuracy: accuracy only on the subset of examples where correction was allowed to fire.")
print("  trigger_coverage: fraction of examples where the hybrid was allowed to intervene.")
print("  net_gain: corrected errors minus new harms caused by the hybrid.")
print()
print(f"Frozen motif artifact : {FROZEN_MOTIF_ARTIFACT}")
print(f"Joint motif artifact  : {JOINT_MOTIF_ARTIFACT}")
print(f"Device                : {device}")
print(f"Output directory      : {OUTPUT_DIR}")

_PROGRESS_STATE = {}

def _progress_logger(**event):
    checkpoint_tag = event.get("checkpoint_tag", "?")
    experiment = event.get("experiment", "?")
    stage = event.get("stage", "?")
    completed = event.get("completed")
    total = event.get("total")
    elapsed = event.get("elapsed_seconds")
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    key = (checkpoint_tag, experiment, stage)
    should_print = False
    bucket = None
    if completed is None or total is None:
        should_print = True
    else:
        total = max(int(total), 1)
        completed = int(completed)
        if completed <= 1 or completed >= total:
            should_print = True
        elif total <= 10:
            should_print = True
        else:
            bucket = int((100 * completed) / total) // 20
        if bucket is not None and _PROGRESS_STATE.get(key) != bucket:
            should_print = True
            _PROGRESS_STATE[key] = bucket
    if not should_print:
        return
    parts = [f"[{checkpoint_tag}]", experiment, stage]
    if completed is not None and total is not None:
        parts.append(f"{completed}/{total}")
    if elapsed is not None:
        parts.append(f"elapsed={elapsed:.0f}s")
    if eta is not None:
        parts.append(f"eta={eta:.0f}s")
    if message:
        parts.append(message)
    print(" | ".join(parts), flush=True)

def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def _run_or_load(path: Path, label: str, fn):
    if path.exists() and not FORCE_RERUN:
        print(f"  [cache hit] {label}: {path.name}", flush=True)
        return _load_json(path)
    print(f"  [compute] {label}: {path.name}", flush=True)
    result = fn(path)
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"  [saved] {label}: {path.name}", flush=True)
    return result

CLEAN_RESULTS = {}
for tag, motif_artifact in motif_artifacts.items():
    print("\n" + "-" * 72)
    print(f"Branch: {tag}")
    checkpoint_path = motif_artifact["metadata"]["checkpoint_path"]
    print(f"Checkpoint: {Path(checkpoint_path).name}")
    components = load_components_from_checkpoint(checkpoint_path, device)
    branch_dir = OUTPUT_DIR / tag
    branch_dir.mkdir(parents=True, exist_ok=True)
    print("Artifacts:")
    for artifact_name in ["clean_utility.json", "corruption_utility.json"]:
        artifact_path = branch_dir / artifact_name
        status = "exists" if artifact_path.exists() else "missing"
        print(f"  {status:<7} {artifact_name}")
    print("Running clean hard-example utility analysis...", flush=True)
    CLEAN_RESULTS[tag] = _run_or_load(
        branch_dir / "clean_utility.json",
        "clean utility",
        lambda cache_path, tag=tag, components=components, motif_artifact=motif_artifact: run_motif_clean_utility_experiment(
            components,
            motif_artifact,
            loaders["fit"],
            loaders["val"],
            loaders["test"],
            device=device,
            checkpoint_tag=tag,
            fit_max_images=FIT_MAX_IMAGES,
            val_max_images=VAL_MAX_IMAGES,
            test_max_images=TEST_MAX_IMAGES,
            top_pairs=HARD_PAIRS_TOP_K,
            trigger_mode=TRIGGER_MODE,
            margin_quantile=MARGIN_QUANTILE,
            top_motif_fraction=TOP_MOTIF_FRACTION,
            min_top_motifs=MIN_TOP_MOTIFS,
            max_top_motifs=MAX_TOP_MOTIFS,
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )


In [ ]:
CORRUPTION_RESULTS = {}
for tag, motif_artifact in motif_artifacts.items():
    print("\n" + "-" * 72)
    print(f"Corruption branch: {tag}")
    checkpoint_path = motif_artifact["metadata"]["checkpoint_path"]
    components = load_components_from_checkpoint(checkpoint_path, device)
    branch_dir = OUTPUT_DIR / tag
    print("Running corruption transfer analysis with clean motifs...", flush=True)
    CORRUPTION_RESULTS[tag] = _run_or_load(
        branch_dir / "corruption_utility.json",
        "corruption utility",
        lambda cache_path, tag=tag, components=components, motif_artifact=motif_artifact: run_motif_corruption_utility_experiment(
            components,
            motif_artifact,
            device=device,
            checkpoint_tag=tag,
            data_dir=base_config["data"]["data_dir"],
            batch_size=base_config["data"]["batch_size"],
            corruption_names=CORRUPTION_NAMES,
            severities=CORRUPTION_SEVERITIES,
            fit_max_images=FIT_MAX_IMAGES,
            val_max_images=VAL_MAX_IMAGES,
            test_max_images=TEST_MAX_IMAGES,
            top_pairs=HARD_PAIRS_TOP_K,
            trigger_mode=TRIGGER_MODE,
            margin_quantile=MARGIN_QUANTILE,
            top_motif_fraction=TOP_MOTIF_FRACTION,
            min_top_motifs=MIN_TOP_MOTIFS,
            max_top_motifs=MAX_TOP_MOTIFS,
            num_workers=base_config["data"].get("num_workers", 4),
            seed=base_config["data"].get("seed", 0),
            augment_fit=base_config["data"].get("augment_fit", True),
            download=base_config["data"].get("download", True),
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )


In [ ]:
print("=" * 72)
print("Final Summary")
print("=" * 72)
print("Clean utility:")
for tag in ("frozen", "joint"):
    summary = CLEAN_RESULTS[tag]["summary"]
    print(f"[{tag}] backbone={summary['backbone_overall_accuracy']:.4f} | full_motif={summary['full_motif_overall_accuracy']:.4f} | top_motif={summary['top_motif_overall_accuracy']:.4f} | trigger_acc={summary['top_motif_trigger_accuracy']:.4f} | trigger_coverage={summary['trigger_coverage']:.4f} | top_motif_net_gain={summary['top_motif_net_gain']}")
print("\nCorruption utility:")
for tag in ("frozen", "joint"):
    summary = CORRUPTION_RESULTS[tag]["summary"]
    print(f"[{tag}] mean_backbone={summary['mean_backbone_overall_accuracy']:.4f} | mean_full_motif={summary['mean_full_motif_overall_accuracy']:.4f} | mean_top_motif={summary['mean_top_motif_overall_accuracy']:.4f} | full_gain={summary['mean_full_motif_gain_vs_backbone']:+.4f} | top_gain={summary['mean_top_motif_gain_vs_backbone']:+.4f}")

joint_clean = CLEAN_RESULTS["joint"]["summary"]
frozen_clean = CLEAN_RESULTS["frozen"]["summary"]
joint_corr = CORRUPTION_RESULTS["joint"]["summary"]
frozen_corr = CORRUPTION_RESULTS["frozen"]["summary"]

print("\nPlain-English interpretation:")
if joint_clean["top_motif_overall_accuracy"] > frozen_clean["top_motif_overall_accuracy"]:
    print("- On clean hard examples, the joint branch gives the stronger compact motif correction signal.")
elif joint_clean["top_motif_overall_accuracy"] < frozen_clean["top_motif_overall_accuracy"]:
    print("- On clean hard examples, the frozen branch still gives the stronger compact motif correction signal.")
else:
    print("- On clean hard examples, the frozen and joint compact motif hybrids are about tied.")

if joint_corr["mean_top_motif_gain_vs_backbone"] > frozen_corr["mean_top_motif_gain_vs_backbone"]:
    print("- Under corruption, the joint branch transfers better: its compact motif subset gives the larger average gain over the backbone.")
elif joint_corr["mean_top_motif_gain_vs_backbone"] < frozen_corr["mean_top_motif_gain_vs_backbone"]:
    print("- Under corruption, the frozen branch transfers better: its compact motif subset gives the larger average gain over the backbone.")
else:
    print("- Under corruption, frozen and joint branches are giving very similar compact motif gains.")

if joint_corr["mean_top_motif_gain_vs_backbone"] > 0 or frozen_corr["mean_top_motif_gain_vs_backbone"] > 0:
    print("- At least one branch shows that clean motif families remain useful when inputs are corrupted, which is a strong sign that the motifs are not just overfit to clean images.")
else:
    print("- Neither branch is clearly beating the backbone under corruption yet, so the motifs may still be more descriptive than useful.")

print("\nWhat to look at next in the saved JSON outputs:")
print("- pair_rows in clean_utility.json: tells you which hard class pairs the motif hybrid helps or hurts.")
print("- rows in corruption_utility.json: tells you which corruption types/severities benefit most from motif transfer.")
